In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from meta_analytics.dataframes import get_screening_df
from django_pandas.io import read_frame
from meta_rando.models import RandomizationList
from edc_appointment.analytics import get_appointment_df
from edc_pdutils.dataframes import get_subject_visit
from edc_registration.models import RegisteredSubject
from meta_consent.models import SubjectConsent

In [ ]:
df_screening = get_screening_df()

In [ ]:
qs = RandomizationList.objects.values("subject_identifier", "allocated_datetime").all()
df_rando = read_frame(qs, verbose=False)


In [ ]:
df_appt = get_appointment_df()
df_appt = df_appt[df_appt["visit_code"]==1000.0][["subject_identifier", "appt_datetime", "timepoint_datetime"]].copy().reset_index(drop=True)


In [ ]:
df_visit = get_subject_visit("meta_subject.subjectvisit")


In [ ]:
df_consent = read_frame(SubjectConsent.objects.values("subject_identifier", "consent_datetime").all(), verbose=False)

In [ ]:
df_rs = read_frame(RegisteredSubject.objects.values("subject_identifier", "randomization_datetime", "registration_datetime").all(), verbose=False)

In [ ]:
df = df_screening[df_screening.subject_identifier.str.startswith("105-")].copy().reset_index(drop=True)

In [ ]:
df = df.merge(df_rando[["subject_identifier", "allocated_datetime"]], on="subject_identifier", how="left")

In [ ]:
df = df[["subject_identifier", "report_datetime", "eligibility_datetime", "allocated_datetime"]].rename(columns={"report_datetime": "screening_datetime", "allocated_datetime": "rando_datetime"})


In [ ]:
df = df.merge(df_consent, on="subject_identifier", how="left")

In [ ]:
df = df.merge(df_appt, on="subject_identifier", how="left")

In [ ]:
df = df.merge(df_visit[df_visit["visit_code"]==1000.0][["subject_identifier", "visit_datetime"]], on="subject_identifier", how="left")


In [ ]:
df["diff"] = df["visit_datetime"]-df["rando_datetime"]

In [ ]:
import pandas as pd

def utc_to_local_naive(df: pd.DataFrame, tz: str = "America/Chicago", suffix: str = "datetime") -> pd.DataFrame:
    df = df.copy()
    cols = [c for c in df.columns if c.lower().endswith(suffix)]
    for c in cols:
        df[c] = (
            pd.to_datetime(df[c], utc=True)   # naive -> assumed UTC; tz-aware -> converted to UTC
            .dt.tz_convert(tz)
            .dt.tz_localize(None)             # drop tz, keep local wall-clock time
            .astype("datetime64[ns]")
        )
    return df


In [ ]:
td_cols = df.select_dtypes(include=["timedelta64"]).columns
print(td_cols.tolist())  # see which ones they are

for c in td_cols:
    # df[c] = df[c].dt.total_seconds()        # float; NaT -> NaN -> Stata missing
    # df[c] = df[c] / pd.Timedelta(days=1)   # fractional days
    df[c] = df[c] / pd.Timedelta(hours=1)  # hours

df = utc_to_local_naive(df, tz="Africa/Dar_es_Salaam")  # site's tz, not yours
dt_cols = [c for c in df.columns if c.lower().endswith("datetime")]

df.to_stata(analysis_folder / "meta3_dates.dta", version=118, write_index=False, convert_dates={c: "tc" for c in dt_cols}, variable_labels={c: f"{c} (seconds)" for c in td_cols})

In [ ]:
df